In [ ]:
# imports
import math
import numpy as np
import matplotlib.pyplot as plt
import plotly as plt
import pandas as pd
from datetime import datetime

In [ ]:
# MUST ADD CALIBRATED DATA TO THIS!!!
# use data_processing.ipynb BEFORE USING THIS

In [ ]:
# variables
filename = 'nothing right now'

In [ ]:
# constants
c_drag = 0.1
r = 0.25
velocity = 15
mass = 50/1000
g = 9.8
rho_air = 1.225
rho_helium = 0.179
mu = 0.0000181
# mu is dynamic viscosity of air
# theta is angle from vertical downward direction
# beta is angle in horizontal plane
theta = math.radians(55)
beta1 = 0
beta2 = math.pi/2
beta3 = math.pi
beta4 = 3 * math.pi/2

In [ ]:
# first, calculating buoyancy, and lift

volume = (4/3) * math.pi * r**3
m_helium = rho_helium * volume
# Solving buoyancy and weight
f_buoyancy = (rho_air) * g * volume
# add mass of helium
f_weight = (mass + m_helium) * g
# add lift
f_lift = np.array([[0], [0], [f_buoyancy - f_weight]])

In [ ]:
# loading data
df = pd.read_csv(filename, header=None, names=['time', 'T0', 'T1', 'T2', 'T3'])

In [ ]:
# Calculating Tension and Drag

vectorized_df = pd.DataFrame(columns = ['time', 'Tension0', 'Tension1', 'Tension2', 'Tension3'])

def tension_vector(vectorized_df, theta, beta):
    ''' vectorizes the tension in a new df'''
    vectorized_df['Tension0'] = np.array([
        [df['T0'] * math.sin(theta) * math.cos(beta)],
        [df['T0'] * math.sin(theta) * math.sin(beta)],
        [-df['T0'] * math.cos(theta)]
    ])
    df['Tension1'] = np.array([
        [df['T1'] * math.sin(theta) * math.cos(beta)],
        [df['T1'] * math.sin(theta) * math.sin(beta)],
        [-df['T1'] * math.cos(theta)]
    ])
    vectorized_df['Tension2'] = np.array([
        [df['T2'] * math.sin(theta) * math.cos(beta)],
        [df['T2'] * math.sin(theta) * math.sin(beta)],
        [-df['T2'] * math.cos(theta)]
    ])
    vectorized_df['Tension3'] = np.array([
        [df['T3'] * math.sin(theta) * math.cos(beta)],
        [df['T3'] * math.sin(theta) * math.sin(beta)],
        [-df['T3'] * math.cos(theta)]
    ])
    return vectorized_df

def drag_vector(vectorized_df, f_lift):
    ''' creates a new column in vectorized_df for drag, 
    which is the net horizontal force due to tension and lift'''
    # finding net horizontal force due to tension
    total_tension = vectorized_df['Tension0'] + vectorized_df['Tension1'] + vectorized_df['Tension2'] + vectorized_df['Tension3']
    vectorized_df['drag'] = -total_tension + f_lift

    # finding the direction of the drag for each time
    vectorized_df['drag_direction'] = vectorized_df['drag'] / np.linalg.norm(vectorized_df['drag'])

    # finding the magnitude of the drag for each time
    vectorized_df['drag_magnitude'] = np.linalg.norm(vectorized_df['drag'])
    return vectorized_df['drag'], vectorized_df['drag_direction'], vectorized_df['drag_magnitude']


In [ ]:
# velocity vector

def velocity_mag (vectorized_df, c_drag, rho_air, r):
    ''' finds the magnitude of the velocity'''
    vectorized_df['velocity'] = math.sqrt((2 * vectorized_df['drag_magnitude']) / (c_drag * rho_air * math.pi * r**2))
    return vectorized_df['velocity']

def velocity_vector (vectorized_df):
    if 'drag_direction' not in vectorized_df.columns:
        raise ValueError("drag_direction column is missing. Please run drag_vector function first.")
    else:
        vectorized_df['velocity_vector'] = vectorized_df['velocity'] * vectorized_df['drag_direction'] / vectorized_df['drag_magnitude']
    return vectorized_df['velocity_vector']

def find_angles(vectorized_df):
    vectorized_df['alpha'] = math.atan2(vectorized_df['drag'][1], vectorized_df['drag'][0])
    vectorized_df['phi'] = math.atan2(vectorized_df['drag'][2], math.sqrt(vectorized_df['drag'][0]**2 + vectorized_df['drag'][1]**2))
    return vectorized_df[['alpha']], vectorized_df[['phi']]



In [ ]:
# graph velocity magnitude over time
